# Reproduce the Berry--Esseen parameter bank on an A100

Before running the notebook:

1. In **Runtime > Change runtime type**, select an **A100 GPU**.
2. Add two Colab secrets and grant this notebook access to them:
   - `BERRY_ESSEEN_REPOSITORY`: the Git repository URL.
   - `BERRY_ESSEEN_REF`: the complete 40-hex-character commit SHA to reproduce. Branches and tags are deliberately rejected.
3. Choose **Runtime > Run all** and authorize Google Drive when prompted.

Checkpoints, unique per-run logs, provenance records, and completed content-addressed parameter banks are mirrored to `MyDrive/berry_esseen_parameter_bank_reproduction`. A completed bank becomes visible through `published_parameter_bank.json` only after every file has been copied and authenticated. A disconnected runtime can resume by running the notebook again with the same commit SHA and a matching recorded build profile; incompatible checkpoints are ignored and recomputed.


In [ ]:
from datetime import datetime, timezone
from importlib import metadata as importlib_metadata
from pathlib import Path
import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import sys

from google.colab import drive, userdata

try:
    REPOSITORY_URL = (userdata.get("BERRY_ESSEEN_REPOSITORY") or "").strip()
    REPOSITORY_REF = (userdata.get("BERRY_ESSEEN_REF") or "").strip()
except Exception as exc:
    raise RuntimeError(
        "Grant this notebook access to the BERRY_ESSEEN_REPOSITORY and "
        "BERRY_ESSEEN_REF Colab secrets."
    ) from exc
if not REPOSITORY_URL:
    raise RuntimeError("BERRY_ESSEEN_REPOSITORY must be non-empty.")
if re.fullmatch(r"[0-9a-fA-F]{40}", REPOSITORY_REF) is None:
    raise RuntimeError(
        "BERRY_ESSEEN_REF must be a complete 40-hex-character commit SHA; "
        "branches, abbreviated hashes, and tags are not accepted."
    )
REQUESTED_COMMIT = REPOSITORY_REF.lower()
if sys.version_info[:2] != (3, 12):
    raise RuntimeError(
        f"The locked GPU wheels require CPython 3.12; detected {platform.python_version()}."
    )
if sys.platform != "linux" or platform.machine().lower() not in {"x86_64", "amd64"}:
    raise RuntimeError(
        f"The reproduction profile requires Linux x86_64; detected {sys.platform} {platform.machine()}."
    )

drive.mount("/content/drive")

REPOSITORY_DIR = Path("/content/berry_esseen_repository")
LOCAL_OUTPUT = Path("/content/berry_esseen_parameter_bank_build")
DRIVE_MIRROR = Path("/content/drive/MyDrive/berry_esseen_parameter_bank_reproduction")
DRIVE_MIRROR.mkdir(parents=True, exist_ok=True)

if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
git_environment = os.environ.copy()
git_environment["GIT_LFS_SKIP_SMUDGE"] = "1"
subprocess.run(
    ["git", "clone", "--filter=blob:none", "--no-checkout", REPOSITORY_URL, str(REPOSITORY_DIR)],
    env=git_environment,
    check=True,
)
subprocess.run(
    ["git", "-C", str(REPOSITORY_DIR), "fetch", "--depth=1", "origin", REQUESTED_COMMIT],
    env=git_environment,
    check=True,
)
subprocess.run(
    ["git", "-C", str(REPOSITORY_DIR), "-c", "advice.detachedHead=false", "checkout", "--detach", "FETCH_HEAD"],
    env=git_environment,
    check=True,
)
RESOLVED_COMMIT = subprocess.check_output(
    ["git", "-C", str(REPOSITORY_DIR), "rev-parse", "HEAD"],
    text=True,
).strip().lower()
if RESOLVED_COMMIT != REQUESTED_COMMIT:
    raise RuntimeError(
        f"Requested commit {REQUESTED_COMMIT}, but Git resolved {RESOLVED_COMMIT}."
    )
if subprocess.check_output(
    ["git", "-C", str(REPOSITORY_DIR), "status", "--porcelain", "--untracked-files=no"],
    text=True,
).strip():
    raise RuntimeError("The checked-out tracked repository files are not clean.")

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_DIR = DRIVE_MIRROR / "runs" / f"{RESOLVED_COMMIT}_{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=False)
BUILD_LOG = RUN_DIR / "build.log"
PROVENANCE_PATH = RUN_DIR / "provenance.json"

gpu_requirements = REPOSITORY_DIR / "requirements-gpu.txt"
if not gpu_requirements.is_file():
    raise FileNotFoundError(gpu_requirements)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", "--force-reinstall", "-r", str(gpu_requirements)],
    check=True,
)
print(f"Repository commit: {RESOLVED_COMMIT}")


In [ ]:
import cupy as cp
import numpy as np
import pandas as pd

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def atomic_write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
    os.replace(temporary, path)

device_id = cp.cuda.Device().id
properties = cp.cuda.runtime.getDeviceProperties(device_id)
gpu_name_value = properties["name"]
GPU_NAME = gpu_name_value.decode() if isinstance(gpu_name_value, bytes) else str(gpu_name_value)
COMPUTE_CAPABILITY = (int(properties["major"]), int(properties["minor"]))
if "A100" not in GPU_NAME.upper():
    raise RuntimeError(f"An NVIDIA A100 is required; allocated device: {GPU_NAME}")
if COMPUTE_CAPABILITY != (8, 0):
    raise RuntimeError(
        f"A100 compute capability 8.0 is required; detected {COMPUTE_CAPABILITY}."
    )

CUDA_RUNTIME = int(cp.cuda.runtime.runtimeGetVersion())
CUDA_DRIVER = int(cp.cuda.runtime.driverGetVersion())
cuda_major = CUDA_RUNTIME // 1000
cuda_minor = (CUDA_RUNTIME % 1000) // 10
if cuda_major != 12:
    raise RuntimeError(f"CUDA runtime 12.x is required; detected {cuda_major}.{cuda_minor}.")
installed_distributions = sorted(
    f"{dist.metadata.get('Name', 'unknown')}=={dist.version}"
    for dist in importlib_metadata.distributions()
)
pip_freeze = sorted(
    subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze", "--all"], text=True
    ).splitlines()
)
LOCKED_DISTRIBUTIONS = (
    "cupy-cuda12x", "cuda-pathfinder", "numpy", "pandas",
    "python-dateutil", "pytz", "six", "tzdata",
)

def distribution_fingerprint(name):
    distribution = importlib_metadata.distribution(name)
    files = distribution.files or []
    result = {"name": distribution.metadata["Name"], "version": distribution.version}
    for filename in ("METADATA", "WHEEL", "RECORD"):
        matches = [file for file in files if file.name == filename]
        if len(matches) != 1:
            raise RuntimeError(
                f"Installed distribution {name!r} has {len(matches)} {filename} files."
            )
        result[f"{filename.lower()}_sha256"] = sha256_file(
            distribution.locate_file(matches[0])
        )
    return result

locked_distribution_fingerprints = {
    name: distribution_fingerprint(name) for name in LOCKED_DISTRIBUTIONS
}
SOURCE_PATHS = {
    "builder": REPOSITORY_DIR / "reproduction" / "build_parameter_bank.py",
    "reproduction_package": REPOSITORY_DIR / "reproduction" / "__init__.py",
    "gpu_search": REPOSITORY_DIR / "reproduction" / "gpu_search.py",
    "refinement_plan": REPOSITORY_DIR / "reproduction" / "publication_refinement_plan.npz",
    "reference_manifest": REPOSITORY_DIR / "data" / "parameter_bank" / "manifest.json",
    "gpu_requirements": REPOSITORY_DIR / "requirements-gpu.txt",
}
PROVENANCE = {
    "format": "berry-esseen-a100-reproduction-run-v1",
    "status": "environment-validated",
    "run_id": RUN_ID,
    "requested_commit": REQUESTED_COMMIT,
    "resolved_commit": RESOLVED_COMMIT,
    "repository_url_sha256": hashlib.sha256(REPOSITORY_URL.encode("utf-8")).hexdigest(),
    "python": platform.python_version(),
    "python_implementation": platform.python_implementation(),
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "cupy": cp.__version__,
    "cuda_runtime": CUDA_RUNTIME,
    "cuda_driver": CUDA_DRIVER,
    "gpu_name": GPU_NAME,
    "compute_capability": list(COMPUTE_CAPABILITY),
    "gpu_total_memory_bytes": int(properties["totalGlobalMem"]),
    "source_sha256": {name: sha256_file(path) for name, path in SOURCE_PATHS.items()},
    "installed_distributions": installed_distributions,
    "installed_distributions_sha256": hashlib.sha256(("\n".join(installed_distributions) + "\n").encode("utf-8")).hexdigest(),
    "pip_freeze": pip_freeze,
    "pip_freeze_sha256": hashlib.sha256(("\n".join(pip_freeze) + "\n").encode("utf-8")).hexdigest(),
    "locked_distribution_fingerprints": locked_distribution_fingerprints,
}
atomic_write_json(PROVENANCE_PATH, PROVENANCE)
print(f"GPU: {GPU_NAME} (compute capability {COMPUTE_CAPABILITY[0]}.{COMPUTE_CAPABILITY[1]})")
print(f"Python: {platform.python_version()}")
print(f"CuPy: {cp.__version__}")
print(f"CUDA runtime: {cuda_major}.{cuda_minor} ({CUDA_RUNTIME}); driver API: {CUDA_DRIVER}")
print(f"Run provenance: {PROVENANCE_PATH}")


In [ ]:
build_command = [
    sys.executable,
    "-m",
    "reproduction.build_parameter_bank",
    "--output-dir",
    str(LOCAL_OUTPUT),
    "--mirror-dir",
    str(DRIVE_MIRROR),
    "--refinement-plan",
    str(REPOSITORY_DIR / "reproduction" / "publication_refinement_plan.npz"),
    "--reference-manifest",
    str(REPOSITORY_DIR / "data" / "parameter_bank" / "manifest.json"),
]
PROVENANCE["status"] = "build-started"
PROVENANCE["build_started_utc"] = datetime.now(timezone.utc).isoformat()
atomic_write_json(PROVENANCE_PATH, PROVENANCE)
print(f"Starting deterministic A100 build. Detailed log: {BUILD_LOG}")
with BUILD_LOG.open("x", encoding="utf-8", buffering=1) as log_stream:
    completed = subprocess.run(
        build_command,
        cwd=REPOSITORY_DIR,
        stdout=log_stream,
        stderr=subprocess.STDOUT,
        text=True,
    )
PROVENANCE["build_finished_utc"] = datetime.now(timezone.utc).isoformat()
PROVENANCE["build_returncode"] = int(completed.returncode)
PROVENANCE["build_log_sha256"] = sha256_file(BUILD_LOG)
if completed.returncode != 0:
    PROVENANCE["status"] = "build-failed"
    atomic_write_json(PROVENANCE_PATH, PROVENANCE)
    log_tail = "\n".join(BUILD_LOG.read_text(encoding="utf-8").splitlines()[-40:])
    raise RuntimeError(f"Parameter-bank reproduction failed. Log tail:\n{log_tail}")
PROVENANCE["status"] = "build-completed"
atomic_write_json(PROVENANCE_PATH, PROVENANCE)
print("Parameter-bank build completed.")


In [ ]:
repository_text = str(REPOSITORY_DIR)
if repository_text not in sys.path:
    sys.path.insert(0, repository_text)
from construction.parameter_bank import ParameterBank, sha256_file as bank_sha256_file
from reproduction.build_parameter_bank import directory_digest

POINTER_PATH = DRIVE_MIRROR / "published_parameter_bank.json"
pointer = json.loads(POINTER_PATH.read_text(encoding="utf-8"))
if pointer.get("format") != "berry-esseen-parameter-bank-publication-v1":
    raise RuntimeError("Unrecognized parameter-bank publication pointer.")
mirror_root = DRIVE_MIRROR.resolve()
ARTIFACT_PATH = (DRIVE_MIRROR / pointer["artifact"]).resolve()
if ARTIFACT_PATH != mirror_root and mirror_root not in ARTIFACT_PATH.parents:
    raise RuntimeError("Published parameter-bank path escapes the Drive mirror.")
REFERENCE_MANIFEST = REPOSITORY_DIR / "data" / "parameter_bank" / "manifest.json"
bank = ParameterBank(ARTIFACT_PATH)
bank.validate()
reference_content = json.loads(REFERENCE_MANIFEST.read_text(encoding="utf-8"))
if bank.manifest != reference_content:
    raise RuntimeError("Reproduced manifest content differs from the repository manifest.")
artifact_manifest_sha256 = bank_sha256_file(ARTIFACT_PATH / "manifest.json")
reference_manifest_sha256 = bank_sha256_file(REFERENCE_MANIFEST)
if artifact_manifest_sha256 != reference_manifest_sha256:
    raise RuntimeError("Reproduced manifest SHA-256 differs from the repository manifest.")
if pointer.get("manifest_sha256") != artifact_manifest_sha256:
    raise RuntimeError("Publication pointer has the wrong manifest SHA-256.")
artifact_tree_sha256 = directory_digest(ARTIFACT_PATH)
if pointer.get("tree_sha256") != artifact_tree_sha256:
    raise RuntimeError("Published parameter-bank tree failed authentication.")

PROVENANCE.update({
    "status": "validated",
    "validated_utc": datetime.now(timezone.utc).isoformat(),
    "artifact_relative_path": pointer["artifact"],
    "artifact_manifest_sha256": artifact_manifest_sha256,
    "artifact_tree_sha256": artifact_tree_sha256,
})
atomic_write_json(PROVENANCE_PATH, PROVENANCE)
print("Parameter bank validated successfully.")
print(f"Rows: {len(bank):,}; candidates per cell: {bank.candidates_per_cell}")
print(f"Artifact: {ARTIFACT_PATH}")
print(f"Manifest SHA-256: {artifact_manifest_sha256}")
print(f"Repository commit: {RESOLVED_COMMIT}")
print(f"Provenance: {PROVENANCE_PATH}")
